# Adding New Robotics Datasets to VLA Foundry

VLA Foundry preprocesses robotics datasets into WebDataset tar files for efficient training. Raw data can come from any source (simulation logs, real robot recordings, HDF5, Parquet, NPZ, MCAP, etc.), as long as you write a **converter** that knows how to read it and produce a standardized output. This tutorial walks through every step.

## Overview

| Step | Objective | File(s) |
|----------|-----------|----------|
| 0 | Understand the output format | current tutorial |
| 1 | Write a Converter class | `vla_foundry/data/preprocessing/robotics/converters/my_format.py` |
| 2 | Register the converter | `vla_foundry/data/preprocessing/robotics/converters/__init__.py`, `vla_foundry/data/preprocessing/robotics/preprocess_params.py` |
| 3 | Run preprocessing | CLI or Ray cluster |

---

## Step 0: Understanding the Output Format

The preprocessing pipeline converts raw episodes into **WebDataset tar shards** that the training data loaders consume directly. Understanding this output format is essential before writing a converter.

### Directory structure

After preprocessing, the output directory will contain:

```
output_directory/
├── frames/
│   ├── {unique_id}_{episode_id}_frame_{frame_idx}.tar   # One tar per sample (intermediate)
│   └── ...
├── shards/
│   ├── shard_000000.tar          # Shuffled training shards
│   ├── shard_000001.tar
│   ├── ...
│   ├── manifest.jsonl            # Shard names and sample counts
│   ├── stats.json                # Dataset-level statistics (mean, std, min, max per lowdim key)
│   ├── preprocessing_config.yaml # Copy of config used
│   ├── processing_metadata.json  # Timing, sample counts, git info
│   └── COMPLETED                 # Marker file (written only by the Ray-based data_generation_ray.py pipeline, not by the standalone preprocess_robotics_to_tar.py script)
└── episodes/
    ├── {episode_id}.tar          # All samples from one episode, in order
    ├── manifest.jsonl
    └── stats.json
```

The `shards/` directory is what the training pipeline reads. The `episodes/` directory provides episode-ordered access for evaluation and visualization.

### Contents of each shard

Each tar shard contains multiple samples. Each sample is a set of files sharing a UUID prefix:

```
shard_000000.tar
├── {uuid}.lowdim.npz                      # Low-dimensional data
├── {uuid}.{camera_name}_t{offset}.jpg     # RGB images (one per camera per time offset)
├── {uuid}.language_instructions.json       # Language task description
├── {uuid}.metadata.json                    # Sample metadata
└── ...
```

### The `lowdim.npz` file

This is a NumPy compressed archive containing all low-dimensional arrays for the sample. It **must** contain:

- `past_mask`: boolean array of shape `[T]` where `T = past_lowdim_steps + future_lowdim_steps + 1`. Marks which timesteps are valid past observations (not padded).
- `future_mask`: boolean array of shape `[T]`. Marks which timesteps are valid future targets (not padded).

Beyond these required masks, it will contain all the lowdim arrays your converter produces. Common examples:

- Action arrays (e.g., `arm_action`, `base_action`) with shape `[T, D]`
- State/proprioception arrays (e.g., `joint_positions`, `eef_pose`) with shape `[T, D]`
- Camera calibration arrays (e.g., `original_intrinsics.cam1`, `extrinsics.cam1`)

All lowdim arrays share the same time dimension `T` along axis 0.

### The windowing concept

Each sample is built from an **anchor timestep** within an episode. The preprocessing window extends:

- `past_lowdim_steps` timesteps **before** the anchor
- `future_lowdim_steps` timesteps **after** the anchor (inclusive of the anchor)

This produces a lowdim window of size `T = past_lowdim_steps + future_lowdim_steps + 1`.

When the window extends beyond episode boundaries, edge values are **padded** (using the configured `padding_strategy`: `copy`, `zero`, or `reflect`), and the masks indicate which positions are real vs. padded.

Images are extracted at specific offsets from the anchor timestep (configured via `image_indices`, default `[-1, 0]`), meaning "one frame before" and "at the anchor". Images are clamped to valid episode indices rather than padded.

### The `language_instructions.json` file

A JSON dictionary with keys drawn from the set `["original", "randomized", "verbose", "alternative"]`, each mapping to a string describing the task.

### The `metadata.json` file

A JSON dictionary containing sample-level metadata such as:

- `camera_names`: list of cameras present in this sample
- `anchor_relative_idx`: index of the anchor timestep within the lowdim window (equals `past_lowdim_steps`)
- Any additional metadata from your converter (episode ID, timestamps, etc.)

---

## Step 1: Write a Converter Class

### 1a. The Base Class

All converters inherit from `BaseRoboticsConverter` defined in `vla_foundry/data/preprocessing/robotics/converters/base.py`. The base class:

- Receives the config (`cfg: PreprocessParams`) in its constructor
- Handles the full processing pipeline via `process_episode()` -- you do **not** need to override this method
- Calls your extraction methods for each episode, then for each anchor timestep
- Manages parallel upload of samples via `ThreadPoolExecutor`
- Batches statistics updates to the Ray statistics actor

### 1b. Required Methods

You **must** implement these methods. The exact signatures (from `base.py`) are shown below.

#### 1. `discover_episodes`

In [ ]:
# Create synthetic demo episodes so the cells below can run end-to-end.
# In practice, your source data already lives on disk.
import os
import tempfile

import numpy as np

T, H, W = 20, 64, 64  # 20 timesteps, 64×64 images
CAMERA_NAMES = ["front_camera", "wrist_camera"]
rng = np.random.default_rng(42)

demo_dir = tempfile.mkdtemp(prefix="vla_tutorial_")
for i in range(3):
    np.savez(
        os.path.join(demo_dir, f"episode_{i:03d}.npz"),
        front_camera=rng.integers(0, 255, (T, H, W, 3), dtype=np.uint8),
        wrist_camera=rng.integers(0, 255, (T, H, W, 3), dtype=np.uint8),
        actions=rng.standard_normal((T, 7)).astype(np.float32),
        joint_positions=rng.standard_normal((T, 7)).astype(np.float32),
        task_description=np.array(f"Pick up object {i}"),
    )

files = sorted(os.listdir(demo_dir))
print(f"Created {len(files)} synthetic episodes in {demo_dir}:")
for f in files:
    size = os.path.getsize(os.path.join(demo_dir, f))
    print(f"  {f}  ({size / 1024:.1f} KB)")

In [ ]:
# Signature:
# def discover_episodes(self, source_paths: list[str], max_episodes_to_process: int = -1) -> list[str]:
#
# Given a list of source directory paths, return a flat list of episode paths.
# Each path is later passed to load_episode_data().
# If max_episodes_to_process > 0, truncate the list accordingly.


def discover_episodes_example(source_paths, max_episodes_to_process=-1):
    all_episodes = []
    for source_dir in source_paths:
        for fname in sorted(os.listdir(source_dir)):
            if fname.endswith(".npz"):
                all_episodes.append(os.path.join(source_dir, fname))
    all_episodes.sort()
    if max_episodes_to_process > 0:
        all_episodes = all_episodes[:max_episodes_to_process]
    return all_episodes


episodes = discover_episodes_example([demo_dir])
print(f"Discovered {len(episodes)} episodes:")
for ep in episodes:
    print(f"  {os.path.basename(ep)}")

#### 2. `load_episode_data`

In [ ]:
# Signature:
# def load_episode_data(self, episode_path: str) -> Any:
#
# Load a single episode from disk and return it in whatever format is convenient.
# The return value is passed unchanged to get_episode_length(), extract_camera_data(),
# extract_lowdim_data(), etc.


def load_episode_data_example(episode_path):
    data = np.load(episode_path, allow_pickle=True)
    return data


episode_data = load_episode_data_example(episodes[0])

print(f"Loaded: {os.path.basename(episodes[0])}")
print(f"\n{'Key':<25} {'Shape':<22} {'Dtype'}")
print("-" * 55)
for key in episode_data.files:
    arr = episode_data[key]
    print(f"{key:<25} {str(arr.shape):<22} {arr.dtype}")

#### 3. `get_episode_length`

In [ ]:
# Signature:
# def get_episode_length(self, episode_data: Any) -> int:
#
# Return the number of timesteps in the episode.


def get_episode_length_example(episode_data):
    return len(episode_data["actions"])


length = get_episode_length_example(episode_data)
print(f"Episode length: {length} timesteps")

#### 4. `extract_camera_data`

In [ ]:
# Signature:
# def extract_camera_data(self, episode_data: Any):
#
# Return a dictionary mapping camera names to image data covering all timesteps.
# Values can be:
#   - A list/array of numpy images (H, W, 3) -- will be JPEG-encoded during upload
#   - A list of raw bytes -- will be written as-is (must already be JPEG-encoded)
# Return None to skip camera data.

import matplotlib.pyplot as plt


def extract_camera_data_example(episode_data, camera_names):
    camera_data = {}
    for camera_name in camera_names:
        camera_data[camera_name] = episode_data[camera_name]  # shape: (T, H, W, 3)
    return camera_data


camera_data = extract_camera_data_example(episode_data, CAMERA_NAMES)

print("Camera data:")
for name, imgs in camera_data.items():
    print(f"  {name}: {imgs.shape}  ({imgs.dtype})")

fig, axes = plt.subplots(1, len(CAMERA_NAMES), figsize=(6, 3))
for ax, (name, imgs) in zip(axes, camera_data.items(), strict=False):
    ax.imshow(imgs[0])
    ax.set_title(f"{name}\n(t=0)")
    ax.axis("off")
plt.suptitle("Sample frames from episode 0", y=1.02)
plt.tight_layout()
plt.show()

#### 5. `extract_lowdim_data`

In [ ]:
# Signature:
# def extract_lowdim_data(self, episode_data: Any):
#
# Return a dictionary mapping lowdim key names to numpy arrays of shape (T, D)
# covering all timesteps. These arrays will be windowed and padded in extract_sample_data().


def extract_lowdim_data_example(episode_data):
    return {
        "actions": episode_data["actions"],  # (T, 7)
        "joint_positions": episode_data["joint_positions"],  # (T, 7)
    }


lowdim_data = extract_lowdim_data_example(episode_data)

print("Lowdim data:")
print(f"  {'Key':<22} {'Shape':<15} {'Dtype':<10} {'Min':>8}  {'Max':>8}")
print("  " + "-" * 68)
for key, arr in lowdim_data.items():
    print(f"  {key:<22} {str(arr.shape):<15} {str(arr.dtype):<10} {arr.min():>8.3f}  {arr.max():>8.3f}")

print(f"\nFirst timestep — actions: {lowdim_data['actions'][0]}")

#### 6. `extract_sample_data`

```python
def extract_sample_data(
    self,
    anchor_timestep: int,
    episode_path: str,
    episode_length: int,
    camera_data: dict[str, Any],
    lowdim_data: dict[str, Any],
    intrinsics_data: dict[str, Any],
    extrinsics_data: dict[str, Any],
    metadata_data: dict[str, Any],
    statistics_ray_actor,
    logger_actor,
):
```

This is the core method. For a given anchor timestep, extract the windowed sample from the full-episode data. Returns a tuple of:

```
(sample_images, sample_lowdim, sample_metadata, language_instructions, [stats_sample])
```

- `sample_images`: dict mapping `"{camera_name}_t{offset}"` to image data (numpy array or bytes)
- `sample_lowdim`: dict mapping lowdim key names to windowed arrays of shape `[T, D]`, **must include** `past_mask` and `future_mask`
- `sample_metadata`: dict with at least `camera_names` and `anchor_relative_idx`
- `language_instructions`: dict with at least an `"original"` key
- Optional 5th element: stats sample dict or `None` (for batched statistics updates)

Returning `(None, None, None, None)` signals that this sample should be **filtered out** (e.g., too much padding, still sample).

### 1c. Optional Methods

These methods have default implementations in the base class but can be overridden:

#### `extract_intrinsics_extrinsics_data`

```python
def extract_intrinsics_extrinsics_data(self, episode_data: Any):
    """Return (intrinsics_data, extrinsics_data) or (None, None)."""
    return None, None
```

Override if your dataset includes camera calibration data. Return dicts mapping camera names to arrays of shape `(T, 3, 3)` for intrinsics and `(T, 4, 4)` for extrinsics.

#### `extract_metadata_data`

```python
def extract_metadata_data(self, episode_data: Any):
    """Return a dict of global metadata or per-timestep metadata arrays."""
    return None
```

Override to provide additional metadata (e.g., timestamps, episode index, depth scale).

#### `get_episode_id`

```python
def get_episode_id(self, episode_path: str) -> str:
    """Get episode ID from episode path. Default: basename of the path."""
    return os.path.basename(episode_path.rstrip("/"))
```

Override if you need a custom episode naming scheme (e.g., to avoid collisions when using multiple source directories).

### 1d. Complete Example Converter

Below is a minimal but complete converter for a hypothetical format where each episode is a single NPZ file containing:
- `images`: array of shape `(T, H, W, 3)` with uint8 RGB images
- `actions`: array of shape `(T, D)` with float32 actions
- `joint_positions`: array of shape `(T, J)` with float32 joint states
- `task_description`: a string describing the task

This example is closely modeled on the LeRobot converter (the simplest production converter) but further simplified.

In [ ]:
import inspect
from typing import Any

try:
    from vla_foundry.data.preprocessing.robotics.converters.base import BaseRoboticsConverter
    from vla_foundry.data.preprocessing.robotics.preprocess_masks import create_past_and_future_masks
    from vla_foundry.data.preprocessing.utils import is_still_sample

    _deps_available = True
except ImportError:
    # Preprocessing dependencies not installed yet — run cell 27 (`uv sync`) first.
    # The class definition below is still valid reference code.
    _deps_available = False
    BaseRoboticsConverter = object


class SimpleNPZConverter(BaseRoboticsConverter):
    """Converter for the SimpleNPZ format."""

    def __init__(self, cfg):
        super().__init__(cfg)

    # ------------------------------------------------------------------
    # Episode discovery and loading
    # ------------------------------------------------------------------

    def discover_episodes(self, source_paths: list[str], max_episodes_to_process: int = -1) -> list[str]:
        all_episodes = []
        for source_dir in source_paths:
            for fname in sorted(os.listdir(source_dir)):
                if fname.endswith(".npz"):
                    all_episodes.append(os.path.join(source_dir, fname))
        all_episodes.sort()
        if max_episodes_to_process > 0:
            all_episodes = all_episodes[:max_episodes_to_process]
        return all_episodes

    def load_episode_data(self, episode_path: str) -> dict:
        data = np.load(episode_path, allow_pickle=True)
        return {key: data[key] for key in data.files}

    def get_episode_length(self, episode_data: dict) -> int:
        return len(episode_data["actions"])

    # ------------------------------------------------------------------
    # Full-episode extraction (called once per episode)
    # ------------------------------------------------------------------

    def extract_camera_data(self, episode_data: dict) -> dict[str, np.ndarray]:
        camera_data = {}
        for camera_name in self.cfg.camera_names:
            camera_data[camera_name] = episode_data[camera_name]  # (T, H, W, 3)
        return camera_data

    def extract_lowdim_data(self, episode_data: dict) -> dict[str, np.ndarray]:
        return {
            "actions": episode_data["actions"],  # (T, D)
            "joint_positions": episode_data["joint_positions"],  # (T, J)
        }

    def extract_metadata_data(self, episode_data: dict) -> dict:
        return {
            "task_description": str(episode_data.get("task_description", "")),
        }

    # ------------------------------------------------------------------
    # Per-sample extraction (called once per anchor timestep)
    # ------------------------------------------------------------------

    def extract_sample_data(
        self,
        anchor_timestep: int,
        episode_path: str,
        episode_length: int,
        camera_data: dict[str, np.ndarray],
        lowdim_data: dict[str, np.ndarray],
        intrinsics_data: dict[str, Any],
        extrinsics_data: dict[str, Any],
        metadata_data: dict[str, Any],
        statistics_ray_actor,
        logger_actor,
    ):
        # 1. Track total potential samples
        logger_actor.increment_total_potential_samples.remote()

        # 2. Calculate the lowdim window boundaries
        lowdim_start = anchor_timestep - self.cfg.past_lowdim_steps
        lowdim_end = anchor_timestep + self.cfg.future_lowdim_steps

        # 3. Determine how much padding is needed
        past_padding = max(0, -lowdim_start)
        future_padding = max(0, lowdim_end - episode_length + 1)

        # 4. Filter: skip if padding exceeds allowed limits
        if past_padding > self.cfg.max_padding_left or future_padding > self.cfg.max_padding_right:
            logger_actor.increment_padding_samples_filtered.remote()
            return None, None, None, None

        valid_start = max(0, lowdim_start)
        valid_end = min(episode_length - 1, lowdim_end)

        # 5. Filter: skip still/stationary samples
        if self.cfg.filter_still_samples and is_still_sample(
            lowdim_data, valid_start, valid_end, self.cfg.still_threshold
        ):
            logger_actor.increment_still_samples_filtered.remote()
            return None, None, None, None

        # 6. Extract images at the configured offsets
        sample_images = {}
        for img_offset in self.image_indices:  # e.g., [-1, 0]
            img_timestep = int(np.clip(anchor_timestep + img_offset, 0, episode_length - 1))
            for camera_name, camera_images in camera_data.items():
                key = f"{camera_name}_t{img_offset}"
                sample_images[key] = camera_images[img_timestep]

        # 7. Window and pad lowdim data
        sample_lowdim = {}
        for key, data in lowdim_data.items():
            valid_data = data[valid_start : valid_end + 1]
            if past_padding > 0 or future_padding > 0:
                valid_data = self.pad_fn(valid_data, past_padding, future_padding)
            sample_lowdim[key] = valid_data

        # 8. Create past/future masks
        past_mask, future_mask = create_past_and_future_masks(
            anchor_timestep, self.cfg.past_lowdim_steps, self.cfg.future_lowdim_steps, episode_length
        )

        # 9. Build stats_sample BEFORE adding masks to sample_lowdim
        #    (masks should not be included in dataset statistics)
        stats_sample = (
            None
            if statistics_ray_actor is None
            else {
                "lowdim": {k: v.copy() for k, v in sample_lowdim.items()},
                "past_mask": past_mask,
                "future_mask": future_mask,
            }
        )

        # 10. Add masks to sample_lowdim (after building stats_sample)
        sample_lowdim["past_mask"] = past_mask
        sample_lowdim["future_mask"] = future_mask

        # 11. Build sample metadata
        sample_metadata = {
            "camera_names": list(camera_data.keys()),
            "anchor_relative_idx": int(self.cfg.past_lowdim_steps),
        }
        if metadata_data:
            for key, value in metadata_data.items():
                sample_metadata[key] = value

        # 12. Language instructions
        language_instructions = {
            "original": metadata_data.get("task_description", "Perform the task.")
            if metadata_data
            else "Perform the task."
        }

        return sample_images, sample_lowdim, sample_metadata, language_instructions, None, None, stats_sample


# Show the public interface of the converter
public_methods = [
    (name, m)
    for name, m in inspect.getmembers(SimpleNPZConverter, predicate=inspect.isfunction)
    if not name.startswith("_")
]
print("SimpleNPZConverter — public methods:")
for name, m in public_methods:
    sig = inspect.signature(m)
    params = list(sig.parameters.keys())[1:]  # drop 'self'
    print(f"  {name}({', '.join(params)})")

if not _deps_available:
    print("\n⚠  vla_foundry preprocessing deps not installed — run the `uv sync` cell in Step 3 first.")

Key patterns to note:

1. **Always call `logger_actor.increment_total_potential_samples.remote()`** at the top of `extract_sample_data()`.
2. **Always call `logger_actor.increment_padding_samples_filtered.remote()`** or `logger_actor.increment_still_samples_filtered.remote()` when filtering, before returning `(None, None, None, None)`.
3. **Build `stats_sample` before adding masks** to `sample_lowdim`. The statistics computation uses the masks to distinguish valid vs. padded data, so they must be separate.
4. **Use `self.pad_fn()`** (set by `PaddingStrategy.get_pad_fn()` in the base class) for consistent padding behavior.
5. **Use `self.image_indices`** (set from `cfg.image_indices` in the base class, defaults to `[-1, 0]`) for image time offsets.
6. **Use `np.clip()`** for image timesteps -- images are clamped to valid bounds rather than padded.
7. **The return tuple** has 4 required elements and up to 3 optional ones: `(images, lowdim, metadata, language, stats_sample?)`.

---

## Step 2: Register the Converter

### 2a. Add to `converters/__init__.py`

Open `vla_foundry/data/preprocessing/robotics/converters/__init__.py` and add an `elif` branch to the `get_converter()` function:

```python
def get_converter(cfg: PreprocessParams) -> BaseRoboticsConverter:
    if cfg.type == "spartan":
        from vla_foundry.data.preprocessing.robotics.converters.spartan import SpartanConverter

        return SpartanConverter(cfg)
    elif cfg.type == "lerobot":
        from vla_foundry.data.preprocessing.robotics.converters.lerobot import LeRobotConverter

        return LeRobotConverter(cfg)
    # ... existing converters ...
    elif cfg.type == "simple_npz":
        from vla_foundry.data.preprocessing.robotics.converters.simple_npz import SimpleNPZConverter

        return SimpleNPZConverter(cfg)
    else:
        raise ValueError(f"Unsupported source type: {cfg.type}")
```

Note: The import is inside the `elif` block (lazy import) to avoid loading unnecessary dependencies.

### 2b. Add a PreprocessParams subclass

Open `vla_foundry/data/preprocessing/robotics/preprocess_params.py` and add a params dataclass for your format. Use the `@register_preprocess_params` decorator to register it with draccus:

```python
@register_preprocess_params("simple_npz")
@dataclass(frozen=True)
class SimpleNPZPreprocessParams(PreprocessParams):
    """Preprocessing params for the SimpleNPZ format."""

    # Add any format-specific fields here. For example:
    # observation_keys: list[str] = field(default=None)
    # action_keys: list[str] = field(default=None)
    pass
```

The `@register_preprocess_params("simple_npz")` decorator:
1. Registers the class with draccus's `ChoiceRegistry` so `--type simple_npz` works.
2. Sets the internal `_type` attribute so `cfg.type` returns `"simple_npz"`.

If your format needs format-specific parameters (like the LeRobot converter needs `observation_keys` and `action_keys`), add them as dataclass fields with defaults.

### 2c. Add to `TYPE_MAPPER`

At the bottom of `preprocess_params.py`, add your new class to the `TYPE_MAPPER` dictionary:

```python
TYPE_MAPPER = {
    "spartan": SpartanPreprocessParams,
    "lerobot": LeRobotPreprocessParams,
    "simple_npz": SimpleNPZPreprocessParams,
}
```

This is used by the standalone entry point (`preprocess_robotics_to_tar.py`) to resolve `--type` to the correct config class.

---

## Step 3: Run Preprocessing

That's it! Your dataset should now be supported! To verify the implementation, you can try to run the preprocessing. There's a detailed tutorial for this at a separate notebook. Please refer to that notebook for more details.

The standalone entry point is `vla_foundry/data/preprocessing/preprocess_robotics_to_tar.py`. It starts a local Ray instance, discovers episodes, processes them, and creates shards.

In [ ]:
# Install preprocessing dependencies
!uv sync --group preprocessing --group tutorials

```bash
# Run preprocessing
uv run python vla_foundry/data/preprocessing/preprocess_robotics_to_tar.py \
    --type simple_npz \
    --source_episodes "['path/to/episode_dir/']" \
    --output_dir /tmp/my_dataset_output/ \
    --camera_names "['front_camera', 'wrist_camera']" \
    --resize_images_size "[342, 256]" \
    --samples_per_shard 100 \
    --past_lowdim_steps 1 \
    --future_lowdim_steps 14 \
    --max_padding_left 1 \
    --max_padding_right 7 \
    --compute_statistics true \
    --db_logging false \
    --skip_git_tagging true
```

> **Warning**: If you have a stale Ray cluster running, the standalone script may connect to it instead of starting a new local instance. Run `ray stop` before running locally to avoid package version mismatches.

Notes:
- `--source_episodes` accepts a Python list literal as a string.
- `--resize_images_size` is required when camera data is numpy arrays (not pre-encoded JPEG bytes). Format: `[width, height]`.
- `--db_logging false` disables DynamoDB logging (useful for local testing without AWS/DynamoDB access).
- `--skip_git_tagging true` disables automatic git tag creation and push (see note below). Recommended for local testing.
- Ray starts automatically in local mode when no cluster is detected.

> **Note: Auto-tagging behavior.** By default, the preprocessing script inspects the git working tree for uncommitted changes to preprocessing-related files. If it finds critical changes, it automatically creates an annotated git tag (e.g., `preprocess_robotics_data_20260324_153000`) and pushes it to origin. This is useful for reproducibility in production but can be surprising during local development. Set `--skip_git_tagging true` to disable all git operations, or `--auto_tag false` to disable only the automatic tag creation while still recording git info in metadata.